In [0]:
"""
STEP 3: LOAD USERS & TRANSACTIONS INTO DATABRICKS
Create Delta Lake tables from CSV files

This creates:
- banking_agent_db.users table
- Updates banking_agent_db.transactions with user_id column

Run this in your Databricks notebook
"""

import pandas as pd
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType, TimestampType

print("="*70)
print("STEP 3: LOAD INTO DATABRICKS")
print("="*70)

# ============================================
# ENSURE DATABASE EXISTS
# ============================================

print("\n[DB] Ensuring banking_agent_db exists...")

spark.sql("CREATE DATABASE IF NOT EXISTS banking_agent_db")
spark.sql("USE banking_agent_db")

print("  ✓ Database ready")

# ============================================
# CREATE USERS TABLE
# ============================================

print("\n[USERS] Creating users table...")

try:
    # Read users CSV
    users_df = pd.read_csv("/Workspace/Repos/molugurikoushik68@gmail.com/banking-agent/banking-agent/data/users.csv")

    print(f"  ✓ Loaded {len(users_df)} users from CSV")

    # Convert to Spark DataFrame
    users_spark_df = spark.createDataFrame(users_df)

    # Create or replace table
    users_spark_df.write.mode("overwrite").saveAsTable("banking_agent_db.users")

    print(f"  ✓ Created table: banking_agent_db.users")

    # Verify
    count = spark.sql("SELECT COUNT(*) as count FROM banking_agent_db.users").collect()[0]['count']
    print(f"  ✓ Verified: {count} users in table")

except Exception as e:
    print(f"  ⚠️ Error creating users table: {e}")
    print(f"  Creating fallback users table...")

    # Fallback: Create empty table
    schema = StructType([
        StructField("user_id", StringType()),
        StructField("name", StringType()),
        StructField("email", StringType()),
        StructField("balance", DoubleType()),
        StructField("account_status", StringType()),
        StructField("kyc_verified", BooleanType()),
        StructField("created_at", TimestampType())
    ])

    empty_df = spark.createDataFrame([], schema=schema)
    empty_df.write.mode("overwrite").saveAsTable("banking_agent_db.users")
    print(f"  ✓ Created empty users table (fallback)")

# ============================================
# UPDATE TRANSACTIONS WITH USER_ID
# ============================================

print("\n[TRANSACTIONS] Updating transactions with user_id...")

try:
    # Read transactions CSV (already has user_id from Step 2)
    transactions_df = pd.read_csv("/Workspace/Repos/molugurikoushik68@gmail.com/banking-agent/banking-agent/data/sample_transactions.csv")

    print(f"  ✓ Loaded {len(transactions_df)} transactions from CSV")
    print(f"  ✓ Columns: {list(transactions_df.columns)}")

    # Convert to Spark DataFrame
    transactions_spark_df = spark.createDataFrame(transactions_df)

    # Create or replace table
    transactions_spark_df.write.mode("overwrite").saveAsTable("banking_agent_db.transactions")

    print(f"  ✓ Updated table: banking_agent_db.transactions")

    # Verify
    count = spark.sql("SELECT COUNT(*) as count FROM banking_agent_db.transactions").collect()[0]['count']
    print(f"  ✓ Verified: {count} transactions in table")

    # Show distribution
    dist = spark.sql("""
    SELECT user_id, COUNT(*) as count FROM banking_agent_db.transactions
    GROUP BY user_id ORDER BY user_id
    """).collect()

    print(f"  ✓ Transaction distribution by user:")
    for row in dist:
        print(f"      {row['user_id']}: {row['count']} transactions")

except Exception as e:
    print(f"  ⚠️ Error updating transactions: {e}")

# ============================================
# VERIFY TABLES
# ============================================

print("\n[VERIFY] Checking tables...")

try:
    # Check users table
    users_check = spark.sql("SELECT * FROM banking_agent_db.users LIMIT 3").collect()
    print(f"\n  ✓ Users table sample:")
    for row in users_check:
        print(f"      {row['user_id']}: {row['name']} (Balance: ₹{row['balance']:,})")

    # Check transactions table
    tx_check = spark.sql("SELECT * FROM banking_agent_db.transactions LIMIT 3").collect()
    print(f"\n  ✓ Transactions table sample:")
    for row in tx_check:
        print(f"      {row['transaction_id']}: {row['user_id']} - ₹{row['amount']:.2f}")

except Exception as e:
    print(f"  ⚠️ Verification error: {e}")

# ============================================
# DISPLAY TABLE INFO
# ============================================

print("\n[INFO] Table schemas:")

try:
    print(f"\n  Users table:")
    spark.sql("DESC TABLE banking_agent_db.users").show(truncate=False)

    print(f"\n  Transactions table:")
    spark.sql("DESC TABLE banking_agent_db.transactions").show(truncate=False)
except:
    pass

# ============================================
# SUMMARY
# ============================================

print("\n" + "="*70)
print("✓ STEP 3 COMPLETE")
print("="*70)

print(f"""
Databricks Tables Created:
✓ banking_agent_db.users
✓ banking_agent_db.transactions (with user_id)

Next Step: STEP_4_ADD_LOAD_USER_DATA_METHOD.py
  (Add load_user_data() method to BankingAgentOrchestrator)
""")

# ============================================
# QUICK TEST: Load user data
# ============================================

print("\n[TEST] Quick test - load user_001 data...")

try:
    user_result = spark.sql("""
    SELECT * FROM banking_agent_db.users WHERE user_id = 'user_001'
    """).collect()[0]

    tx_result = spark.sql("""
    SELECT COUNT(*) as count, SUM(amount) as total
    FROM banking_agent_db.transactions WHERE user_id = 'user_001'
    """).collect()[0]

    print(f"""
  User: {user_result['name']}
  Email: {user_result['email']}
  Balance: ₹{user_result['balance']:,.2f}
  Status: {user_result['account_status']}
  KYC: {user_result['kyc_verified']}

  Transactions:
    - Count: {tx_result['count']}
    - Total: ₹{tx_result['total']:,.2f}
  """)

except Exception as e:
    print(f"  ⚠️ Test error: {e}")
